# Step 02 — OSM POI & Building Extraction

**Two input building datasets used in this pipeline:**
- `data/input/buildings.gpkg` — **ALKIS-like 3D buildings** (main source: height, function codes, addresses)
- `data/input/osm_buildings.gpkg` (optional) — **OSM building footprints** for gap-filling where ALKIS is missing

**This notebook handles the OSM side:**
1. Clips the regional PBF to the study boundary (osmium)
2. Extracts essential daily-activity POIs
3. Extracts ALL OSM POIs
4. Extracts OSM building footprints — **skipped if `data/input/osm_buildings.gpkg` is already provided**

**Outputs:**
- `data/output/01_essential_pois.gpkg`
- `data/output/01_all_pois.gpkg`
- `data/output/01_all_buildings_osm.gpkg` (either copied from input or extracted from PBF)

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import subprocess
from pyrosm import OSM

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LLM_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
print('Config loaded. CRS:', TARGET_CRS)

Config loaded. CRS: EPSG:25832


## 1. Clip PBF to study boundary using osmium

In [2]:
import tempfile, json

boundary     = gpd.read_file(STUDY_BOUNDARY_FILE)
boundary_wgs = boundary.to_crs(epsg=4326)
geom         = boundary_wgs.geometry.union_all()

with tempfile.NamedTemporaryFile(mode='w', suffix='.geojson', delete=False) as f:
    json.dump({"type": "Feature", "geometry": geom.__geo_interface__, "properties": {}}, f)
    poly_file = f.name

print(f'Clipping polygon (WGS84), bounds: {geom.bounds}')

result = subprocess.run(
    ['osmium', 'extract',
     '-p', poly_file,
     '-s', 'complete_ways',
     '-o', str(CLIPPED_PBF_FILE),
     '--overwrite',
     str(OSM_PBF_FILE)],
    capture_output=True, text=True
)

if result.returncode != 0:
    raise RuntimeError(f'osmium extract failed:\n{result.stderr}')

print(f'Clipped PBF saved → {CLIPPED_PBF_FILE}')

Clipping polygon (WGS84), bounds: (9.994028124759431, 51.63666780334394, 11.085117065736126, 52.832788122535526)
Clipped PBF saved → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\capacity-pipeline-generic\data\output\area_of_study_clipped.pbf


## 2. Extract essential daily-activity POIs from PBF

In [3]:
osm = OSM(str(CLIPPED_PBF_FILE))
essential_pois = osm.get_pois(custom_filter={'shop': ESSENTIAL_SHOP_CATEGORIES})
essential_pois = essential_pois.to_crs(TARGET_CRS)
essential_pois.to_file(ESSENTIAL_POIS_FILE, driver='GPKG')
print(f'Essential POIs: {len(essential_pois):,} → {ESSENTIAL_POIS_FILE}')

Essential POIs: 1,639 → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\capacity-pipeline-generic\data\output\01_essential_pois.gpkg


## 3. Extract ALL OSM POIs from PBF

In [4]:
osm2     = OSM(str(CLIPPED_PBF_FILE))
all_pois = osm2.get_pois()
all_pois = all_pois.to_crs(TARGET_CRS)
all_pois.to_file(ALL_POIS_FILE, layer='pois', driver='GPKG')
print(f'All POIs: {len(all_pois):,} → {ALL_POIS_FILE}')

All POIs: 68,315 → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\capacity-pipeline-generic\data\output\01_all_pois.gpkg


## 4. OSM Building footprints (for ALKIS gap-filling)

If `data/input/osm_buildings.gpkg` already exists (user-provided), it is used directly.
Otherwise, buildings are extracted from the clipped PBF.

In [5]:
# OSM_BUILDINGS_INPUT_FILE is set in config.py
# It can point to a pre-supplied file OR be set to None to force PBF extraction.
_osm_bld_src = OSM_BUILDINGS_INPUT_FILE if (OSM_BUILDINGS_INPUT_FILE and OSM_BUILDINGS_INPUT_FILE.exists()) else None

if _osm_bld_src:
    print(f'Using pre-supplied OSM buildings: {_osm_bld_src.name}')
    osm_bld = gpd.read_file(_osm_bld_src)
    osm_bld = osm_bld.to_crs(TARGET_CRS)
    print(f'Loaded {len(osm_bld):,} OSM buildings from input file')
else:
    print('No pre-supplied OSM buildings found — extracting from PBF...')
    osm3    = OSM(str(CLIPPED_PBF_FILE))
    osm_bld = osm3.get_buildings()
    osm_bld = osm_bld.to_crs(TARGET_CRS)
    print(f'Extracted {len(osm_bld):,} OSM buildings from PBF')

osm_bld.to_file(ALL_BUILDINGS_OSM_FILE, layer='buildings', driver='GPKG')
print(f'OSM buildings saved → {ALL_BUILDINGS_OSM_FILE}')

No pre-supplied OSM buildings found — extracting from PBF...
Extracted 509,794 OSM buildings from PBF
OSM buildings saved → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\capacity-pipeline-generic\data\output\01_all_buildings_osm.gpkg
